In [1]:
!pip install transformers datasets accelerate peft sentencepiece sacrebleu evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


In [1]:
from peft import LoraConfig, get_peft_model
import os
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
from transformers import TrainerCallback
import time
import os


os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "offline"


In [2]:
dataset = load_dataset("csv", data_files="/content/new_sin.csv")
dataset = dataset["train"].train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-1.3B")
model = AutoModelForSeq2SeqLM.from_pretrained( "facebook/nllb-200-1.3B",
        torch_dtype=torch.float32,
        device_map="cuda")
model.to('cuda')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-23): 24 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192, out_features=1024, bias=True)
       

In [3]:
# Find indices of rows with missing English
missing_indices = [i for i, x in enumerate(train_dataset["english"]) if x is None]
print("Rows with missing English:", missing_indices)

# Optionally, view the corresponding Sinhala text
for i in missing_indices:
    print(f"Row {i} - Sinhala: {train_dataset['sinhala'][i]} | English: {train_dataset['english'][i]}")



Rows with missing English: []


In [4]:
source_lang = "sin_Sinh"
target_lang = "eng_Latn"
tokenizer.src_lang = source_lang
tokenizer.tgt_lang = target_lang

max_length = 128

def preprocess(examples):
    inputs = examples["sinhala"]
    targets = examples["english"]
    model_inputs = tokenizer(inputs, max_length=max_length, truncation=True)
    labels = tokenizer(targets, max_length=max_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess, batched=True)
eval_dataset = eval_dataset.map(preprocess, batched=True)


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)


Map:   0%|          | 0/1388 [00:00<?, ? examples/s]

Map:   0%|          | 0/155 [00:00<?, ? examples/s]

In [5]:

training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb_sinhala2english",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=3e-4,
    num_train_epochs=10,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=1000,
    logging_dir="./logs",
    predict_with_generate=True,
    fp16=True,
    gradient_accumulation_steps=2,
    dataloader_pin_memory=True,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    save_total_limit=3,
)

# Load BLEU metric
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    result["bleu"] = result["score"]
    return result

In [6]:
class RealTimeProgressCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        logs = kwargs.get("logs", {})
        if "loss" in logs:
            current_time = time.strftime("%H:%M:%S", time.localtime())
            print(f"[{current_time}] Step: {state.global_step}, Loss: {logs['loss']:.4f}", flush=True)
        if state.global_step % 500 == 0:
            trainer.save_model(f"./nllb_sinhala2english_gpu/checkpoint-{state.global_step}")


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
# Add the callback to the trainer
trainer.add_callback(RealTimeProgressCallback)


trainer.args.logging_steps = 1
trainer.args.eval_steps = 150

trainer.train()

model.save_pretrained("./nllb_sinhala2english_lora")
tokenizer.save_pretrained("./nllb_sinhala2english_lora")


/tmp/ipython-input-3529163034.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss,Validation Loss,Score,Counts,Totals,Precisions,Bp,Sys Len,Ref Len,Bleu
150,1.431200,0.736085,40.590590,"[1383, 860, 574, 378]","[1940, 1785, 1630, 1476]","[71.28865979381443, 48.17927170868347, 35.214723926380366, 25.609756097560975]",0.967549,1940,2004,40.590590
300,0.280600,0.670959,43.747041,"[1414, 907, 628, 434]","[1956, 1801, 1646, 1492]","[72.29038854805727, 50.360910605219324, 38.15309842041312, 29.08847184986595]",0.975759,1956,2004,43.747041
450,1.906300,0.636463,44.877924,"[1424, 923, 642, 459]","[1958, 1803, 1648, 1494]","[72.72727272727273, 51.192457016084305, 38.95631067961165, 30.72289156626506]",0.976780,1958,2004,44.877924
600,1.100500,0.612583,44.726507,"[1437, 924, 637, 452]","[1960, 1805, 1650, 1496]","[73.31632653061224, 51.19113573407202, 38.60606060606061, 30.21390374331551]",0.977801,1960,2004,44.726507
750,0.612100,0.603396,47.435992,"[1468, 975, 691, 496]","[2005, 1850, 1695, 1541]","[73.21695760598504, 52.7027027027027, 40.766961651917406, 32.18689162881246]",1.000000,2005,2004,47.435992
900,0.042800,0.594284,45.835134,"[1475, 973, 690, 506]","[2076, 1921, 1766, 1612]","[71.05009633911368, 50.6507027589797, 39.07134767836919, 31.389578163771713]",1.000000,2076,2004,45.835134
1050,0.228500,0.577307,49.751016,"[1486, 1004, 733, 544]","[2006, 1851, 1696, 1542]","[74.0777666999003, 54.2409508373852, 43.219339622641506, 35.27885862516213]",1.000000,2006,2004,49.751016
1200,0.748700,0.538668,50.852348,"[1506, 1027, 751, 553]","[1981, 1826, 1671, 1517]","[76.02221100454317, 56.24315443592552, 44.94314781567923, 36.45352669742913]",0.988457,1981,2004,50.852348
1350,0.438500,0.525582,51.942026,"[1527, 1046, 768, 572]","[1994, 1839, 1684, 1530]","[76.57973921765296, 56.87873844480696, 45.605700712589076, 37.38562091503268]",0.994998,1994,2004,51.942026
1500,0.844600,0.549195,48.685407,"[1542, 1060, 780, 579]","[2144, 1989, 1834, 1680]","[71.92164179104478, 53.29311211664153, 42.529989094874594, 34.464285714285715]",1.000000,2144,2004,48.685407


('./nllb_sinhala2english_lora/tokenizer_config.json',
 './nllb_sinhala2english_lora/special_tokens_map.json',
 './nllb_sinhala2english_lora/sentencepiece.bpe.model',
 './nllb_sinhala2english_lora/added_tokens.json',
 './nllb_sinhala2english_lora/tokenizer.json')

In [8]:
!zip -r "nllb_sinhala2english_lora.zip" "./nllb_sinhala2english_lora"


  adding: nllb_sinhala2english_lora/ (stored 0%)
  adding: nllb_sinhala2english_lora/adapter_config.json (deflated 56%)
  adding: nllb_sinhala2english_lora/sentencepiece.bpe.model (deflated 51%)
  adding: nllb_sinhala2english_lora/README.md (deflated 66%)
  adding: nllb_sinhala2english_lora/special_tokens_map.json (deflated 79%)
  adding: nllb_sinhala2english_lora/adapter_model.safetensors (deflated 7%)
  adding: nllb_sinhala2english_lora/tokenizer_config.json (deflated 94%)
  adding: nllb_sinhala2english_lora/tokenizer.json (deflated 82%)
